In [ ]:
import pickle
import numpy as np
import umap
import matplotlib.pyplot as plt
from scipy.cluster.hierarchy import linkage, optimal_leaf_ordering, dendrogram
from scipy.spatial.distance import pdist

import matplotlib as mpl

mpl.rcParams['figure.dpi'] = 300
mpl.rcParams['savefig.dpi'] = 300
mpl.rcParams['figure.figsize'] = (6, 4)
mpl.rcParams['font.size'] = 10
mpl.rcParams['axes.linewidth'] = 1.0
mpl.rcParams['pdf.fonttype'] = 42
mpl.rcParams['ps.fonttype'] = 42

with open("wb_alltype_sc_results_dict_with_NZ_ES_norm.pkl", "rb") as f:
    data = pickle.load(f)["results"][5]

NZ, names, nodes, spars = [], [], [], []
for r, info in data.items():
    if r in {"SSp", "VPL", "PERI", "ECT", "PO", "PR"}:
        continue
    mr = info["motif_results"].get("0")
    if mr and (nz := np.asarray(mr["NZ"], float)).shape[0] == 13:
        names.append(r)
        NZ.append(nz)
        nodes.append(mr["node"])
        spars.append(mr["sparsity"])

NZ = np.vstack(NZ)
names = np.array(names)
nodes = np.array(nodes)
spars = np.array(spars, float)

emb = umap.UMAP(
    n_neighbors=15,
    min_dist=0.15,
    metric='cosine',
    random_state=1
).fit_transform(NZ)

Z = optimal_leaf_ordering(linkage(emb, "ward"), pdist(emb))

plt.figure(figsize=(6, 11))

rich_labels = [f"{r} (n={int(n)}, s={s:.3f})" for r, n, s in zip(names, nodes, spars)]

dendrogram(
    Z,
    orientation="left",
    labels=rich_labels,
    leaf_font_size=8.5,
    color_threshold=0,
    above_threshold_color='k'
)

plt.title("Hierarchical clustering (optimal leaf ordering)")
plt.xlabel("Distance")
plt.tight_layout()
plt.savefig("1.hierarchical_clustering_dendrogram.svg",
            bbox_inches='tight', dpi=300, transparent=True)
plt.show()


cluster1_names = [
    "FRP",
    "PL",
    "ILA",
    "AId",
    "AIv",
    "ACAd",
    "ACAv",
    "ORBl",
    "ORBvl",
    "ORBm",
    "MOs",
]

cluster2_names = [
    "POST",
    "PRE",
    "ProS",
    "SUB",
    "CA3",
    "PAR",
    "CA1",
    "DG",
]

cluster3_names = [
    "MOp",
    "SSs",
    "SSp-bfd",
    "SSp-m",
    "SSp-n",
    "SSp-ul",
    "RSPd",
    "RSPv",
    "RSPagl",
    "TEa",
    "ENT",
    "VISp",
    "VISa",
    "VISl",
    "VISli",
]

cluster4_names = [
    "VISam",
    "VISpm",
    "VISrl",
    "VISpl",
    "VISpor",
    "VISal",
    "SSp-ll",
    "SSp-tr",
    "AUDp",
    "AUDv",
    "AUDd",
    "AUDpo",
    "PIR",
    "VISC",
    "GU",
    "AIp",
]

custom_order_names = (
    cluster1_names +
    cluster2_names +
    cluster3_names +
    cluster4_names
)

name_to_idx = {r: i for i, r in enumerate(names)}

missing = [r for r in custom_order_names if r not in name_to_idx]
extra   = [r for r in names if r not in custom_order_names]

if missing:
    print("[WARN] These regions were not found in the dataset and will be skipped:", missing)
if extra:
    print("[WARN] These regions are not in the requested order and will be appended at the end:", extra)

ordered_names_list = [r for r in custom_order_names if r in name_to_idx]
ordered_names_list += [r for r in names if r not in ordered_names_list]

region_order = ordered_names_list

order = np.array([name_to_idx[r] for r in region_order], dtype=int)


name_to_cluster = {}
for r in cluster1_names:
    name_to_cluster[r] = 1
for r in cluster2_names:
    name_to_cluster[r] = 2
for r in cluster3_names:
    name_to_cluster[r] = 3
for r in cluster4_names:
    name_to_cluster[r] = 4

labels = np.array([name_to_cluster.get(r, 0) for r in names])

NZ, names, nodes, spars, labels, emb = [np.asarray(arr)[order] for arr in [NZ, names, nodes, spars, labels, emb]]


from adjustText import adjust_text
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(6, 5))


colors = ["#F18D00","#1f77b4","#2ca02c","#d62728"]
for c in sorted(np.unique(labels)):
    lab = "Unassigned" if c == 0 else f"Cluster {c}"
    m = labels == c
    plt.scatter(emb[m, 0], emb[m, 1], label=lab, s=60, c=colors[c-1] ,alpha=0.85, edgecolors="white", linewidths=0.6)

texts = []
for (x, y), n in zip(emb, names):
    texts.append(plt.text(x, y, n, fontsize=7, ha="center", va="center"))

adjust_text(
    texts,
    x=emb[:, 0], y=emb[:, 1],
    expand_points=(1.2, 1.4),
    expand_text=(1.2, 1.4),
    force_points=0.15,
    force_text=0.25,
    # arrowprops=dict(arrowstyle='-', lw=0.4, color='0.4', alpha=0.7),
)

plt.legend(frameon=False)
plt.title("UMAP of brain regions (custom cluster order)")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.tight_layout()
plt.savefig("Fig s4b.svg", bbox_inches="tight", dpi=300, transparent=True)
plt.show()
plt.figure(figsize=(9, 8))
im = plt.imshow(np.nan_to_num(np.log10(1 + 8*NZ), nan=-100.0, posinf=-100.0, neginf=-100.0), aspect='auto', cmap='bwr', vmin=-1, vmax=1)
plt.colorbar(im, label="NZ-score")

plt.yticks(range(len(names)), names, fontsize=8)
plt.xticks(range(13), [f"M{i+1}" for i in range(13)], rotation=45)
plt.title("log10(1+8*NZ) heatmap (custom region order)")
plt.tight_layout()
plt.savefig("3.nz_score_heatmap.svg",
            bbox_inches='tight', dpi=300, transparent=True)
plt.show()

In [ ]:
X = NZ[:, :12]                                # (n_region, 12)
norms = np.linalg.norm(X, axis=1, keepdims=True)
X_norm = X / norms

cos_sim = X_norm @ X_norm.T
cmaps = [
    "RdBu_r"
    # # perceptually uniform
    # "viridis","cividis","plasma","inferno","magma","cubehelix",
    # # restrained sequential
    # "Blues","YlGnBu","PuBuGn","Purples","Greens","OrRd","YlOrBr","BuPu",
    # # high-contrast / heat-ish
    # "turbo","hot","afmhot","gist_heat","bone",
    # # diverging (optional)
    # "coolwarm","RdBu_r","BrBG","PuOr",
]
for cmap in cmaps:
    plt.figure(figsize=(7, 6))
    im = plt.imshow(cos_sim, aspect="auto", cmap=cmap, vmin=0, vmax=1)
    plt.colorbar(im, label="Cosine similarity of NZ-score")
    plt.xticks(range(len(names)), names, rotation=90, fontsize=7)
    plt.yticks(range(len(names)), names, fontsize=7)
    plt.title(f"Cosine similarity matrix (cmap={cmap})")
    plt.tight_layout()
    plt.savefig(f"4.cosine_similarity_matrix_{cmap}.svg", bbox_inches="tight", dpi=300, transparent=True)
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

unique_clusters = np.unique(labels)
n_clust = len(unique_clusters)
colors = plt.cm.tab10(np.arange(n_clust) % 10)

motif_groups = [
    (list(range(0, 4)),  "Motifs 1–4"),
    (list(range(4, 8)),  "Motifs 5–8"),
    (list(range(8, 13)), "Motifs 9–13"),
]

for idx_group, (motif_idx_list, title_group) in enumerate(motif_groups, 1):
    n_motifs = len(motif_idx_list)
    n_cols = 2
    n_rows = int(np.ceil(n_motifs / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6 * n_cols, 4 * n_rows), squeeze=False)

    for k, j in enumerate(motif_idx_list):
        row = k // n_cols
        col = k % n_cols
        ax = axes[row, col]

        vals = NZ[:, j]
        data_by_cluster = [vals[labels == c] for c in unique_clusters]

        # violin
        parts = ax.violinplot(
            data_by_cluster,
            positions=np.arange(1, n_clust + 1),
            showmeans=True,
            showextrema=False,
            showmedians=False,
        )

        for body, col_c in zip(parts['bodies'], colors):
            body.set_facecolor(col_c)
            body.set_edgecolor(col_c)
            body.set_alpha(0.5)

        for i, c in enumerate(unique_clusters, start=1):
            mask = (labels == c)
            vals_c = vals[mask]
            jitter = (np.random.rand(len(vals_c)) - 0.5) * 0.15
            x_pos = np.full(len(vals_c), i) + jitter
            ax.scatter(x_pos, vals_c, s=15, alpha=0.9, color=colors[i-1])

        ax.set_xticks(np.arange(1, n_clust + 1))
        ax.set_xticklabels([f"C{int(c)}" for c in unique_clusters], fontsize=9)
        ax.axhline(0, color="k", lw=1)
        ax.set_ylabel("NZ-score")
        ax.set_title(f"Motif {j+1}")

    for k in range(n_motifs, n_rows * n_cols):
        row = k // n_cols
        col = k % n_cols
        fig.delaxes(axes[row, col])

    fig.suptitle(f"Motif NZ-score across clusters ({title_group})", fontsize=14)
    plt.tight_layout()
    plt.show()

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt

# ===== 0. Basic params =====
pkl_path = "wb_alltype_sc_results_dict_with_NZ_ES_norm.pkl"
param = 5
thr_pick = "0.1"  # choose ONE: "0", "0.1", "0.2", "0.3", "0.4"
regions = ["FRP", "CA1", "MOp", "AUDp"]
regions_label =  ["FRP-PFC", "CA1-Hippo", "MOp-Motor", "AUDp-Sensory"]
# ===== 1. Load =====
with open(pkl_path, "rb") as f:
    results = pickle.load(f)["results"]




# ===== 2. Collect NZ for each region at the same threshold =====
nz_mat = []
valid_regions = []
for r in regions:
    mr = results[param][r]["motif_results"].get(thr_pick, None)
    if mr is None:
        print(f"[WARN] missing: region={r}, thr={thr_pick}")
        continue
    nz = np.asarray(mr["NZ"], float)
    if nz.shape[0] != 13:
        print(f"[WARN] bad NZ shape: region={r}, shape={nz.shape}")
        continue
    nz_mat.append(nz)
    valid_regions.append(r)

nz_mat = np.log10(1+8*np.vstack(nz_mat))  # (n_region, 13)

# ===== 3. Plot grouped bars (4 regions) =====
x = np.arange(13)
bw = 0.18

colors = {
    "MOp-E":   "#601986",
    "MOp":     "#F3CC4F",
    "AVE":     "#009944",
    "FRP":     "#F18D00",
    "FRP-E":   "#E60012",
    "Vanilla": "#529DCB",
    "CA1":     "#00BFC4",
    "AUDp":    "#A6761D"
}
# colors = plt.cm.tab10(np.arange(n_clust) % 10)

# colors = plt.cm.viridis(np.linspace(0.15, 0.85, len(valid_regions)))

plt.figure(figsize=(10, 5))
for i, r in enumerate(valid_regions):
    plt.bar(x + (i - (len(valid_regions)-1)/2)*bw, nz_mat[i], width=bw,
            alpha=0.85, color=colors[r], label=regions_label[i])

plt.xticks(x, [f"M{i}" for i in range(1, 14)], rotation=45)
plt.axhline(0, color="k", lw=1)
plt.ylabel("NZ-score")
plt.xlabel("Motif")
plt.title(f"Motif NZ-score across regions (thr={thr_pick}, param={param} µm)")
plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.legend(title="region")
plt.tight_layout()
plt.savefig(f"8.NZ_regions_FRP_CA1_MOp_AUDp_thr{thr_pick}_param{param}.svg",
            bbox_inches="tight", transparent=True)
plt.show()

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl

# ===== 0. Basic params =====
pkl_path = "wb_alltype_sc_results_dict_with_NZ_ES_norm.pkl"
param = 5
thr_pick = "0.1"  # "0", "0.1", "0.2", "0.3", "0.4"
regions = ["FRP", "CA1", "MOp", "AUDp"]
regions_label = ["FRP-PFC", "CA1-Hippo", "MOp-Motor", "AUDp-Sensory"]

# ===== style for small panels =====
mpl.rcParams["figure.dpi"] = 300
mpl.rcParams["savefig.dpi"] = 300
mpl.rcParams["font.size"] = 8
mpl.rcParams["axes.linewidth"] = 0.6
mpl.rcParams["pdf.fonttype"] = 42
mpl.rcParams["ps.fonttype"] = 42

colors = {
    "FRP":     "#F18D00",
    "CA1":     "#1f77b4",
    "MOp":     "#2ca02c",
    "AUDp":    "#d62728"
}

# colors = plt.cm.tab10(np.arange(n_clust) % 10)

# ===== 1. Load =====
with open(pkl_path, "rb") as f:
    results = pickle.load(f)["results"]

# ===== 2. Collect NZ for each region at the same threshold =====
nz_mat, valid_regions, valid_labels = [], [], []
for r, lab in zip(regions, regions_label):
    mr = results[param][r]["motif_results"].get(thr_pick, None)
    if mr is None:
        print(f"[WARN] missing: region={r}, thr={thr_pick}")
        continue
    nz = np.asarray(mr["NZ"], float)
    if nz.shape != (13,):
        print(f"[WARN] bad NZ shape: region={r}, shape={nz.shape}")
        continue
    nz_mat.append(nz)
    valid_regions.append(r)
    valid_labels.append(lab)

nz_mat = np.log10(1 + 8 * np.vstack(nz_mat))  # (n_region, 13)

# ===== 3. 4 panels: one bar chart per region =====
x = np.arange(13)

ymin = float(np.nanmin(nz_mat))
ymax = float(np.nanmax(nz_mat))
pad = 0.08 * (ymax - ymin) if ymax > ymin else 0.2
ylims = (-0.52, 1.02)

fig_w_cm, fig_h_cm = 18.0, 4.5
fig, axes = plt.subplots(1, len(valid_regions), figsize=(fig_w_cm/2.54, fig_h_cm/2.54), sharey=True)

if len(valid_regions) == 1:
    axes = [axes]

for ax, r, lab, y in zip(axes, valid_regions, valid_labels, nz_mat):
    ax.bar(x, y, width=0.75, color=colors[r], alpha=0.9, linewidth=0)
    ax.axhline(0, color="k", lw=0.6)
    ax.set_title(lab, pad=2)
    # ax.set_xticks(x)
    # ax.set_xticklabels([f"M{i}" for i in range(1, 14)], rotation=45, ha="right")
    ax.grid(axis="y", linestyle="--", alpha=0.25, linewidth=0.5)
    ax.set_ylim(*ylims)
    ax.tick_params(axis="both", which="both", width=0.6, length=2.0)
    for sp in ax.spines.values():
        sp.set_linewidth(0.6)

axes[0].set_ylabel("log10(1+8·NZ)")
fig.suptitle(f"Motif NZ-score (thr={thr_pick}, param={param} µm)", y=1.02)

plt.tight_layout(pad=0.3)
plt.savefig(f"8.NZ_4panels_FRP_CA1_MOp_AUDp_thr{thr_pick}_param{param}.svg",
            bbox_inches="tight", transparent=True)
plt.show()

In [ ]:
colors

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

# ===========================
# Assumed existing:
# NZ     : (n_region, 13)
# labels : (n_region,)  values in {1,2,3,4} (can include 0 if unassigned)
# ===========================

# ----- Figure size (cm -> inch) -----
fig_w_cm, fig_h_cm = 5.3, 4.0
fig_w_in, fig_h_in = fig_w_cm / 2.54, fig_h_cm / 2.54

# ----- Cluster info / colors / test pairs -----
unique_clusters = np.array(sorted(np.unique(labels).tolist()))
n_clust = len(unique_clusters)
colors = plt.cm.tab10(np.arange(n_clust) % 10)

pairs = [(1, 2), (2, 3), (3, 4), (1, 4)]  # C1-C2, C2-C3, C3-C4, C1-C4

for j in [4,5,8,9,10]:
    # ----- 0) transform + drop invalid values (per motif) -----
    raw = 1 + 8 * NZ[:, j]
    mask = np.isfinite(raw) & (raw > 0) & np.isfinite(labels)
    vals = np.log10(raw[mask])
    labj = labels[mask]

    fig, ax = plt.subplots(figsize=(fig_w_in, fig_h_in))

    # ----- 1) violin data by cluster -----
    data_by_cluster = [vals[labj == c] for c in unique_clusters]

    parts = ax.violinplot(
        data_by_cluster,
        positions=np.arange(1, n_clust + 1),
        showmeans=True,
        showextrema=False,
        showmedians=False,
    )

    # violin colors
    for body, col in zip(parts["bodies"], colors):
        body.set_facecolor(col)
        body.set_edgecolor(col)
        body.set_alpha(0.5)

    # mean line styling
    if "cmeans" in parts and parts["cmeans"] is not None:
        parts["cmeans"].set_color("k")
        parts["cmeans"].set_linewidth(1.0)

    # ----- 2) scatter points with jitter -----
    for i_pos, c in enumerate(unique_clusters, start=1):
        vc = vals[labj == c]
        if len(vc) == 0:
            continue
        jitter = (np.random.rand(len(vc)) - 0.5) * 0.15
        x_pos = np.full(len(vc), i_pos) + jitter
        ax.scatter(x_pos, vc, s=10, alpha=0.5, color=colors[i_pos - 1], edgecolors="none")

    # ----- 3) position map for brackets (if you enable stats) -----
    pos_map = {int(c): i for i, c in enumerate(unique_clusters, start=1)}

    # ----- 4) y-range / ticks -----
    if j <= 11:
        y_low, y_high = -0.6,0.6
        ax.set_yticks(np.arange(-0.6, 0.61, 0.3))
    else:
        y_low, y_high = 0.4, 1.1
        ax.set_yticks([0.5, 1.0])

    # ----- 5) significance brackets (disabled by default) -----
    # n_drawn = 0
    # base0, step, h = (0.44, 0.05, 0.018) if j <= 11 else (0.94, 0.07, 0.02)
    # for (a, b) in pairs:
    #     if (a not in pos_map) or (b not in pos_map):
    #         continue
    #     da = vals[labj == a]
    #     db = vals[labj == b]
    #     if (len(da) < 2) or (len(db) < 2):
    #         continue
    #
    #     p = mannwhitneyu(da, db, alternative="two-sided").pvalue
    #
    #     if p < 0.0001: star = "****"
    #     elif p < 0.001: star = "***"
    #     elif p < 0.01: star = "**"
    #     elif p < 0.05: star = "*"
    #     else: continue
    #
    #     x1, x2 = pos_map[a], pos_map[b]
    #     if x1 > x2:
    #         x1, x2 = x2, x1
    #
    #     y = base0 + n_drawn * step
    #     if y + h > y_high - 0.005:
    #         continue
    #
    #     ax.plot([x1, x1, x2, x2], [y, y + h, y + h, y], lw=0.5, c="k")
    #     ax.text((x1 + x2) / 2, y + h, star, ha="center", va="bottom", fontsize=7)
    #     n_drawn += 1

    # ----- 6) fixed y-limits + reference line -----
    ax.set_ylim(y_low, y_high)
    ax.axhline(0, color="0.6", lw=0.5, ls="--")

    # ----- 7) labels / ticks -----
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_title(f"motif_{j+1}", fontsize=8)

    ax.set_xticks(np.arange(1, n_clust + 1))
    ax.tick_params(axis="x", which="both", labelbottom=False)

    # if (j == 0) or (j == 12):
    #     ax.tick_params(axis="y", which="both", labelleft=True)
    # else:
    ax.tick_params(axis="y", which="both", labelleft=False)

    # ----- 8) spine/tick styling -----
    for sp in ax.spines.values():
        sp.set_linewidth(0.5)
    ax.tick_params(axis="both", which="both", width=0.5, length=2.5)

    out_svg = f"motif_{j+1:02d}_violin_0313.svg"
    plt.tight_layout(pad=0.1)
    plt.savefig(out_svg, transparent=True, bbox_inches="tight")
    plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import mannwhitneyu

# ===========================
# Assumed existing:
# NZ     : (n_region, 13)
# labels : (n_region,) values in {1,2,3,4} (can include 0)
# ===========================

# ----- Single panel size (cm -> inch) -----
fig_w_cm, fig_h_cm = 5.3, 4.0
fig_w_in, fig_h_in = fig_w_cm / 2.54, fig_h_cm / 2.54

# ----- Grid layout for 13 motifs -----
n_motif = 13
ncols = 5
nrows = 1  # 5 rows for 13 motifs

# Big figure size (approx: each panel uses fig_w_in x fig_h_in)
big_w = fig_w_in * ncols
big_h = fig_h_in * nrows

# ----- Cluster info / colors / test pairs -----
unique_clusters = np.array(sorted(np.unique(labels).tolist()))
n_clust = len(unique_clusters)
colors = ["#F18D00","#1f77b4","#2ca02c","#d62728"]

# colors = {
#     "MOp":     "#2ca02c",
#     "FRP":     "#F18D00",
#     "CA1":     "#1f77b4",
#     "AUDp":    "#d62728"
# }

pairs = [(1, 2), (2, 3), (3, 4), (1, 4)]

fig, axes = plt.subplots(nrows, ncols, figsize=(big_w, big_h), squeeze=False)
axes = axes.ravel()

i=0
for j in [4,5,8,9,10]:
    ax = axes[i]
    i = i + 1
    # ----- 0) transform + drop invalid values (per motif) -----
    raw = 1 + 8 * NZ[:, j]
    mask = np.isfinite(raw) & (raw > 0) & np.isfinite(labels)
    vals = np.log10(raw[mask])
    labj = labels[mask]

    # ----- 1) violin data by cluster -----
    data_by_cluster = [vals[labj == c] for c in unique_clusters]

    parts = ax.violinplot(
        data_by_cluster,
        positions=np.arange(1, n_clust + 1),
        showmeans=True,
        showextrema=False,
        showmedians=False,
    )

    for body, col in zip(parts["bodies"], colors):
        body.set_facecolor(col)
        body.set_edgecolor(col)
        body.set_alpha(0.5)

    if "cmeans" in parts and parts["cmeans"] is not None:
        parts["cmeans"].set_color("k")
        parts["cmeans"].set_linewidth(1.0)

    # ----- 2) scatter points with jitter -----
    for i_pos, c in enumerate(unique_clusters, start=1):
        vc = vals[labj == c]
        if len(vc) == 0:
            continue
        jitter = (np.random.rand(len(vc)) - 0.5) * 0.15
        x_pos = np.full(len(vc), i_pos) + jitter
        ax.scatter(x_pos, vc, s=10, alpha=0.5, color=colors[i_pos - 1], edgecolors="none")

    # ----- 3) y-range / ticks -----
    if j in (0,1,2,3):
        y_low, y_high = -1, 1
        ax.set_yticks(np.arange(-1, 1.01, 0.4))
    else:
        y_low, y_high = -0.6, 0.6
        ax.set_yticks(np.arange(-0.6, 0.61, 0.3))

    ax.set_ylim(y_low, y_high)
    ax.axhline(0, color="0.6", lw=0.5, ls="--")

    # ----- 4) title / ticks -----
    ax.set_title(f"motif_{j+1}", fontsize=8)
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.set_xticks(np.arange(1, n_clust + 1))
    ax.tick_params(axis="x", which="both", labelbottom=False)
    ax.tick_params(axis="y", which="both", labelleft=False)

    # ----- 5) spine/tick styling -----
    for sp in ax.spines.values():
        sp.set_linewidth(0.5)
    ax.tick_params(axis="both", which="both", width=0.5, length=2.5)

# Hide unused subplots (there will be 15 slots for 13 motifs)
for k in range(n_motif, nrows * ncols):
    axes[k].axis("off")

plt.tight_layout(pad=0.2)
plt.savefig("motif_violin_all_in_one_0313.svg", transparent=True, bbox_inches="tight")
plt.show()

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt

pkl_path   = "wb_alltype_sc_results_dict_with_NZ_ES_norm.pkl"
region = "MOp"
param = 5
thr_list = ["0", "0.1", "0.2", "0.3", "0.4"]

with open(pkl_path, "rb") as f:
    results = pickle.load(f)["results"]   # dict[param][region]["motif_results"][thr]

freq_dict = {}
nz_dict   = {}
es_dict   = {}

for thr in thr_list:
    mr = results[param][region]["motif_results"][thr]

    real = np.asarray(mr["real_motif"], float)
    freq_dict[thr] = real / real.sum()

    # ---- NZ ----
    nz = np.asarray(mr["NZ"], float)
    nz_dict[thr] = nz

    mu_eq  = np.asarray(mr["er_mu_eq"], float)
    es_raw = np.log2(real) - np.log2(mu_eq)
    es_norm = es_raw / np.linalg.norm(es_raw)
    es_dict[thr] = es_norm

x  = np.arange(13)
bw = 0.12

thr_freq = [t for t in thr_list if t in freq_dict]
colors = plt.cm.coolwarm(np.linspace(0.1, 0.9, len(thr_freq)))

plt.figure(figsize=(10, 5))
for i, thr in enumerate(thr_freq):
    plt.bar(x + i*bw, freq_dict[thr], width=bw,
            alpha=0.85, color=colors[i], label=f"thr={thr}")
plt.xticks(x + bw*(len(thr_freq)-1)/2,
           [f"M{i}" for i in range(1, 14)], rotation=45)
plt.ylabel("frequency (count / sum(count))")
plt.xlabel("Motif")
plt.title(f"{region} – motif frequency vs. strength threshold (param={param} µm)")
plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.legend(title="strength")
plt.tight_layout()
plt.savefig(f"8.{region}_motif_frequency_vs_strength.svg",
                 bbox_inches="tight", transparent=True)
plt.show()

thr_nz = [t for t in thr_list if t in nz_dict]
colors = plt.cm.coolwarm(np.linspace(0.1, 0.9, len(thr_nz)))

plt.figure(figsize=(10, 5))
for i, thr in enumerate(thr_nz):
    plt.bar(x + i*bw, nz_dict[thr], width=bw,
            alpha=0.85, color=colors[i], label=f"thr={thr}")
plt.xticks(x + bw*(len(thr_nz)-1)/2,
           [f"M{i}" for i in range(1, 14)], rotation=45)
plt.axhline(0, color="k", lw=1)
plt.ylabel("NZ-score")
plt.xlabel("Motif")
plt.title(f"{region} – motif NZ-score vs. strength threshold (param={param} µm)")
plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.legend(title="strength")
plt.tight_layout()
plt.savefig(f"8.{region}_motif_NZ_vs_strength.svg",
                 bbox_inches="tight", transparent=True)
plt.show()

thr_es = [t for t in thr_list if t in es_dict]
colors = plt.cm.plasma(np.linspace(0.1, 0.9, len(thr_es)))

plt.figure(figsize=(10, 5))
for i, thr in enumerate(thr_es):
    plt.bar(x + i*bw, es_dict[thr], width=bw,
            alpha=0.85, color=colors[i], label=f"thr={thr}")
plt.xticks(x + bw*(len(thr_es)-1)/2,
           [f"M{i}" for i in range(1, 14)], rotation=45)
plt.axhline(0, color="k", lw=1)
plt.ylabel("Enrichment score (ES, log2 ratio, L2 norm)")
plt.xlabel("Motif")
plt.title(f"{region} – motif enrichment (ES) vs. strength threshold (param={param} µm)")
plt.grid(axis="y", linestyle="--", alpha=0.3)
plt.legend(title="strength")
plt.tight_layout()
plt.savefig(f"8.{region}_motif_ES_vs_strength.svg",
                 bbox_inches="tight", transparent=True)
plt.show()

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt

with open("wb_alltype_sc_results_dict_with_NZ_ES_norm.pkl", "rb") as f:
    results_all = pickle.load(f)["results"]

param_list = [3, 5, 10, 15, 20]
thr_use = "0.1"

# ------------------------------------------------------------
# ------------------------------------------------------------
def is_valid_region(results_all, p, r, thr_use="0.1"):
    if p not in results_all:
        return False
    if r not in results_all[p]:
        return False

    info = results_all[p][r]
    if "motif_results" not in info:
        return False
    if thr_use not in info["motif_results"]:
        return False

    mr = info["motif_results"][thr_use]

    real = np.asarray(mr["real_motif"], float)
    mu_mc = np.asarray(mr["er_mu_mc"], float)
    nz = np.asarray(mr["NZ"], float)

    if nz.shape[0] != 13:
        return False
    if np.any(real < 0.9):
        return False
    if np.any(mu_mc[:12] <= 0.):
        return False

    return True

# ------------------------------------------------------------
# ------------------------------------------------------------
common_regions = []

for r in names:
    ok = True
    for p in param_list:
        if not is_valid_region(results_all, p, r, thr_use):
            ok = False
            break
    if ok:
        common_regions.append(r)

print(f"[INFO] common valid regions across all params = {len(common_regions)}")
print(common_regions)

if len(common_regions) == 0:
    raise ValueError("No regions passed the shared filtering criteria across all parameter settings.")

# ------------------------------------------------------------
# ------------------------------------------------------------
fig, axes = plt.subplots(1, len(param_list), figsize=(20, 9), sharex=True, sharey=False)

all_im = None

for i, (ax, p) in enumerate(zip(axes, param_list)):
    NZ_rows = []

    for r in common_regions:
        mr = results_all[p][r]["motif_results"][thr_use]
        nz = np.asarray(mr["NZ"], float)
        NZ_rows.append(nz)

    NZ_mat = np.vstack(NZ_rows)

    show_mat = np.nan_to_num(
        np.log10(1 + 8 * NZ_mat),
        nan=-100.0,
        posinf=-100.0,
        neginf=-100.0
    )

    im = ax.imshow(show_mat, aspect="auto", cmap="bwr", vmin=-1, vmax=1)
    all_im = im

    ax.set_title(f"param={p} µm", fontsize=11)
    ax.set_xticks(range(13))
    ax.set_xticklabels([f"M{i+1}" for i in range(13)], rotation=45, fontsize=8)
    ax.set_xlabel("Motif", fontsize=10)

    ax.set_yticks(range(len(common_regions)))

    if i == 0:
        ax.set_yticklabels(common_regions, fontsize=8)
        ax.tick_params(axis="y", labelleft=True)
        ax.set_ylabel("Brain regions (UMAP order)", fontsize=11)
    else:
        ax.set_yticklabels([])
        ax.tick_params(axis="y", labelleft=False)

cbar = fig.colorbar(all_im, ax=axes, fraction=0.025, pad=0.01)
cbar.set_label("NZ-score", fontsize=10)

fig.suptitle(
    f"NZ-score heatmaps across distance thresholds (thr={thr_use})\n"
    f"(only regions valid in all params)",
    y=0.98,
    fontsize=13
)

plt.subplots_adjust(left=0.32, right=0.98, bottom=0.14, top=0.88, wspace=0.08)

plt.savefig("Fig_s1a_all_params_row_common_regions.svg", bbox_inches="tight", transparent=True)
plt.show()

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# Scatter + Pearson r: NZ(param=p) vs NZ(param=5) at thr="0.1"
# - use existing NZ (no re-computation)
# - same filter as your heatmap: real>=0.9 and er_mu_mc[:12]>0
# - drop points where x<-0.5 or y<-0.5 after log transform
# ============================================================

pkl_path   = "wb_alltype_sc_results_dict_with_NZ_ES_norm.pkl"
thr_use    = "0.1"
param_ref  = 5
param_list = [3, 10, 15, 20]          # compare these to 5um
save_name  = "Fig_s1b_param_vs_5um_thr01.svg"
drop_below = -0.5

with open(pkl_path, "rb") as f:
    results_all = pickle.load(f)["results"]

# ---- build ref map for param=5: region -> NZ(13) (filtered) ----
ref_map = {}
for r in names:
    if param_ref not in results_all or r not in results_all[param_ref]:
        continue
    mr = results_all[param_ref][r].get("motif_results", {}).get(thr_use, None)
    if mr is None:
        continue
    real  = np.asarray(mr.get("real_motif", None), float)
    mu_mc = np.asarray(mr.get("er_mu_mc", None), float)
    nz    = np.asarray(mr.get("NZ", None), float)
    if real is None or mu_mc is None or nz is None or nz.shape != (13,):
        continue
    if np.any(real < 0.9) or np.any(mu_mc[:12] <= 0.0):
        continue
    ref_map[r] = nz

if len(ref_map) == 0:
    raise RuntimeError(f"No valid regions for param={param_ref}, thr={thr_use}.")

# ---- panels: each param vs 5um ----
fig, axes = plt.subplots(1, len(param_list), figsize=(14, 3.6), dpi=220, sharex=True, sharey=True)

for ax, p in zip(axes, param_list):
    cur_map = {}
    if p not in results_all:
        ax.set_title(f"{p}µm vs 5µm\n(param missing)")
        ax.axis("off")
        continue

    for r in names:
        if r not in results_all[p]:
            continue
        mr = results_all[p][r].get("motif_results", {}).get(thr_use, None)
        if mr is None:
            continue
        real  = np.asarray(mr.get("real_motif", None), float)
        mu_mc = np.asarray(mr.get("er_mu_mc", None), float)
        nz    = np.asarray(mr.get("NZ", None), float)
        if real is None or mu_mc is None or nz is None or nz.shape != (13,):
            continue
        if np.any(real < 0.9) or np.any(mu_mc[:12] <= 0.0):
            continue
        cur_map[r] = nz

    common = [r for r in names if (r in ref_map) and (r in cur_map)]
    if len(common) == 0:
        ax.set_title(f"{p}µm vs 5µm\n(no common regions)")
        ax.axis("off")
        continue

    NZ_5  = np.log10(1 + 8*np.vstack([ref_map[r] for r in common]))  # x
    NZ_p  = np.log10(1 + 8*np.vstack([cur_map[r] for r in common]))  # y

    x = NZ_5.flatten()
    y = NZ_p.flatten()
    n_region, n_motif = NZ_5.shape
    motif_ids = np.tile(np.arange(n_motif), n_region)

    mask = np.isfinite(x) & np.isfinite(y) & (x >= drop_below) & (y >= drop_below)
    x, y, motif_ids = x[mask], y[mask], motif_ids[mask]

    R = np.corrcoef(x, y)[0, 1] if x.size > 1 else np.nan

    sc = ax.scatter(x, y, s=10, alpha=0.6, c=motif_ids, cmap="viridis_r")
    ax.plot([-0.5, 1], [-0.5, 1], ls="--", lw=0.8, color="0.4")
    ax.set_title(f"{p}µm vs 5µm\nr={R:.3f}", fontsize=9)
    ax.grid(True, alpha=0.2)

axes[0].set_xlabel("log10(1+8·NZ)  (5 µm)")
axes[0].set_ylabel("log10(1+8·NZ)  (p µm)")
for ax in axes[1:]:
    ax.set_xlabel("log10(1+8·NZ)  (5 µm)")

# cbar = fig.colorbar(sc, ax=axes, fraction=0.025, pad=0.02)
# cbar.set_label("Motif (M1–M13)")
# cbar.set_ticks(range(13))
# cbar.set_ticklabels([f"M{i+1}" for i in range(13)])

plt.suptitle(f"NZ-score correlation: 5 µm vs other distances (thr={thr_use})", y=1.02)
plt.tight_layout()
plt.savefig(save_name, bbox_inches="tight", dpi=300, transparent=True)
plt.show()

print(f"Reference: param={param_ref} µm, thr={thr_use}, valid regions={len(ref_map)}")
for p in param_list:
    if p not in results_all:
        print(f"{p}µm: param missing")
        continue
    cur_map = {}
    for r in names:
        if r not in results_all[p]:
            continue
        mr = results_all[p][r].get("motif_results", {}).get(thr_use, None)
        if mr is None:
            continue
        real  = np.asarray(mr.get("real_motif", None), float)
        mu_mc = np.asarray(mr.get("er_mu_mc", None), float)
        nz    = np.asarray(mr.get("NZ", None), float)
        if real is None or mu_mc is None or nz is None or nz.shape != (13,):
            continue
        if np.any(real < 0.9) or np.any(mu_mc[:12] <= 0.0):
            continue
        cur_map[r] = nz

    common = [r for r in names if (r in ref_map) and (r in cur_map)]
    if len(common) == 0:
        print(f"{p}µm: no common regions")
        continue

    X = np.log10(1 + 8*np.vstack([ref_map[r] for r in common])).flatten()
    Y = np.log10(1 + 8*np.vstack([cur_map[r] for r in common])).flatten()
    m = np.isfinite(X) & np.isfinite(Y) & (X >= drop_below) & (Y >= drop_below)
    r = np.corrcoef(X[m], Y[m])[0, 1] if m.sum() > 1 else np.nan
    print(f"{p}µm vs 5µm: r={r:.3f} (common regions={len(common)}, points kept={m.sum()})")

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt

with open("wb_alltype_sc_results_dict_with_NZ_ES_norm.pkl", "rb") as f:
    results_all = pickle.load(f)["results"]   # dict[param] -> {region -> info}

param_fix = 5
thr_list  = ["0", "0.1", "0.2", "0.3", "0.4"]

# ------------------------------------------------------------
# ------------------------------------------------------------
def is_valid_region(results_all, param_fix, r, thr_use):
    if param_fix not in results_all:
        return False
    if r not in results_all[param_fix]:
        return False

    info = results_all[param_fix][r]
    if "motif_results" not in info:
        return False
    if thr_use not in info["motif_results"]:
        return False

    mr = info["motif_results"][thr_use]

    real = np.asarray(mr["real_motif"], float)
    mu_mc = np.asarray(mr["er_mu_mc"], float)
    nz = np.asarray(mr["NZ"], float)

    if nz.shape[0] != 13:
        return False
    if np.any(real < 0.9):
        return False
    if np.any(mu_mc[:12] <= 0.0):
        return False

    return True

# ------------------------------------------------------------
# ------------------------------------------------------------
common_regions = []

for r in names:
    ok = True
    for thr_use in thr_list:
        if not is_valid_region(results_all, param_fix, r, thr_use):
            ok = False
            break
    if ok:
        common_regions.append(r)

print(f"[INFO] common valid regions across all thresholds = {len(common_regions)}")
print(common_regions)

if len(common_regions) == 0:
    raise ValueError("No regions passed the shared filtering criteria across all threshold settings.")

# ------------------------------------------------------------
# ------------------------------------------------------------
fig, axes = plt.subplots(1, len(thr_list), figsize=(20, 9), sharex=True, sharey=False)

all_im = None

for i, (ax, thr_use) in enumerate(zip(axes, thr_list)):
    NZ_rows = []

    for r in common_regions:
        mr = results_all[param_fix][r]["motif_results"][thr_use]
        nz = np.asarray(mr["NZ"], float)
        NZ_rows.append(nz)

    NZ_mat = np.vstack(NZ_rows)

    show_mat = np.nan_to_num(
        np.log10(1 + 8 * NZ_mat),
        nan=-100.0,
        posinf=-100.0,
        neginf=-100.0
    )

    im = ax.imshow(show_mat, aspect="auto", cmap="bwr", vmin=-1, vmax=1)
    all_im = im

    ax.set_title(f"thr = {thr_use}", fontsize=10)
    ax.set_xticks(range(13))
    ax.set_xticklabels([f"M{i+1}" for i in range(13)], rotation=45, fontsize=8)
    ax.set_xlabel("Motif")

    ax.set_yticks(range(len(common_regions)))
    if i == 0:
        ax.set_yticklabels(common_regions, fontsize=8)
        ax.tick_params(axis="y", labelleft=True)
        ax.set_ylabel("Brain regions (UMAP order)")
    else:
        ax.set_yticklabels([])
        ax.tick_params(axis="y", labelleft=False)

if all_im is not None:
    cbar = fig.colorbar(all_im, ax=axes, fraction=0.025, pad=0.01)
    cbar.set_label("NZ-score")

fig.suptitle(
    f"NZ-score heatmaps across intensity thresholds (param={param_fix} µm)\n"
    f"(only regions valid in all thresholds)",
    y=0.98
)

plt.subplots_adjust(left=0.32, right=0.98, bottom=0.14, top=0.88, wspace=0.08)
plt.savefig("Fig_s5a_all_thresholds_row_common_regions.svg", bbox_inches="tight", transparent=True)
plt.show()

In [ ]:
import pickle
import numpy as np
import matplotlib.pyplot as plt

# ============================================================
# NZ scatter + Pearson r: each pruning thr vs unpruned thr="0"
# - drop points where x<-0.5 or y<-0.5 (after log transform)
# ============================================================

pkl_path  = "wb_alltype_sc_results_dict_with_NZ_ES_norm.pkl"
param_fix = 5
thr_base  = "0"
thr_list  = ["0.1", "0.2", "0.3", "0.4"]
save_name = "Fig s5b.svg"

with open(pkl_path, "rb") as f:
    results = pickle.load(f)["results"][param_fix]

base_map = {}
for r, info in results.items():
    mr = info.get("motif_results", {}).get(thr_base, None)
    if mr is None:
        continue
    nz = np.asarray(mr.get("NZ", None), float)
    if nz is None or nz.shape != (13,):
        continue
    base_map[r] = nz

if len(base_map) == 0:
    raise RuntimeError(f"No NZ found for base thr={thr_base} at param={param_fix}.")

fig, axes = plt.subplots(1, len(thr_list), figsize=(14, 3.6), dpi=220, sharex=True, sharey=True)

for ax, thr in zip(axes, thr_list):
    cur_map = {}
    for r, info in results.items():
        mr = info.get("motif_results", {}).get(thr, None)
        if mr is None:
            continue
        nz = np.asarray(mr.get("NZ", None), float)
        if nz is None or nz.shape != (13,):
            continue
        cur_map[r] = nz

    common = sorted(set(base_map) & set(cur_map))
    if len(common) == 0:
        ax.set_title(f"thr={thr}\n(no common regions)")
        ax.axis("off")
        continue

    NZ_base = np.log10(1 + 8*np.vstack([base_map[r] for r in common]))
    NZ_cur  = np.log10(1 + 8*np.vstack([cur_map[r]  for r in common]))

    x = NZ_base.flatten()
    y = NZ_cur.flatten()

    n_region, n_motif = NZ_base.shape
    motif_ids = np.tile(np.arange(n_motif), n_region)

    # ---- drop invalid + drop points below -0.5 ----
    mask = np.isfinite(x) & np.isfinite(y) & (x >= -0.5) & (y >= -0.5)
    x, y, motif_ids = x[mask], y[mask], motif_ids[mask]

    R = np.corrcoef(x, y)[0, 1] if x.size > 1 else np.nan

    sc = ax.scatter(x, y, s=10, alpha=0.6, c=motif_ids, cmap="viridis_r")
    ax.plot([-0.5, 1], [-0.5, 1], ls="--", lw=0.8, color="0.4")

    ax.set_title(f"thr={thr}\nr={R:.3f}", fontsize=9)
    ax.grid(True, alpha=0.2)

axes[0].set_xlabel(f"NZ-score (thr={thr_base})")
axes[0].set_ylabel("NZ-score (thr)")
for ax in axes[1:]:
    ax.set_xlabel(f"NZ-score (thr={thr_base})")

plt.tight_layout()
plt.savefig(save_name, bbox_inches="tight", dpi=300, transparent=True)
plt.show()

print(f"Base: param={param_fix}, thr={thr_base}, regions={len(base_map)}")
for thr in thr_list:
    cur_map = {}
    for r, info in results.items():
        mr = info.get("motif_results", {}).get(thr, None)
        if mr is None:
            continue
        nz = np.asarray(mr.get("NZ", None), float)
        if nz is None or nz.shape != (13,):
            continue
        cur_map[r] = nz

    common = sorted(set(base_map) & set(cur_map))
    if len(common) == 0:
        print(f"thr={thr}: no common regions")
        continue

    X = np.log10(1 + 8*np.vstack([base_map[r] for r in common])).flatten()
    Y = np.log10(1 + 8*np.vstack([cur_map[r]  for r in common])).flatten()

    m = np.isfinite(X) & np.isfinite(Y) & (X >= -0.5) & (Y >= -0.5)
    r = np.corrcoef(X[m], Y[m])[0, 1] if m.sum() > 1 else np.nan
    print(f"thr={thr}: Pearson r vs thr={thr_base} = {r:.3f} (points kept={m.sum()})")